# TrOCR Re-Fine-tuning — E2E TIR A100 80GB
**Environment:** PyTorch 2.8 · Python 3.12 · CUDA 12.9 · Ubuntu 24.04 (E2E pre-installed)  
**Model:** Already fine-tuned TrOCR → further fine-tuned on 50k synthetic Indian plates  
**Cost:** ~₹180/hr on A100 80GB On-Demand

## Step 0 — Verify Environment


In [1]:
import subprocess, sys, torch

print('=== System ===')
print(subprocess.getoutput('uname -a'))
print(f'Python : {sys.version}')

print('\n=== GPU ===')
print(subprocess.getoutput('nvidia-smi'))

print('\n=== PyTorch ===')
print(f'Version  : {torch.__version__}')
print(f'CUDA     : {torch.version.cuda}')
print(f'GPU avail: {torch.cuda.is_available()}')

if torch.cuda.is_available():
    print(f'GPU name : {torch.cuda.get_device_name(0)}')
    print(f'VRAM     : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
    print(f'BF16     : {torch.cuda.is_bf16_supported()}')
else:
    raise RuntimeError('CUDA not available — stop and check your instance type.')

=== System ===
Linux n-c3477f36-9c47-49f6-b372-8a206c624fcd-0 5.15.0-94-generic #104-Ubuntu SMP Tue Jan 9 15:25:40 UTC 2024 x86_64 x86_64 x86_64 GNU/Linux
Python : 3.12.3 (main, Feb  4 2025, 14:48:35) [GCC 13.3.0]

=== GPU ===
Thu Apr  2 09:25:40 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.127.08             Driver Version: 550.127.08     CUDA Version: 12.9     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100 80GB PCIe          On  |   00000000:41:00.0 Off |                    0 |
| N/A

## Step 1 — Install Dependencies


In [ ]:
import subprocess
result = subprocess.run(
    [
        'pip', 'install', '-q',
        'transformers==4.44.2',
        'datasets==2.20.0',
        'evaluate==0.4.2',
        'accelerate==0.33.0',
        'sentencepiece==0.2.0',
        'jiwer==3.0.4',
        'Pillow==10.4.0',
        'pandas==2.2.2',
        'scikit-learn==1.5.1',
        'gdown==5.2.0',
        'tqdm==4.66.4',
    ],
    capture_output=True, text=True
)
if result.returncode != 0:
    print('STDERR:', result.stderr[-2000:])
    raise RuntimeError('pip install failed — check output above')

print('Dependencies installed. Verifying torch is still intact...')
import torch
assert torch.cuda.is_available(), 'CUDA lost after pip install — restart kernel and re-run'
print(f'torch {torch.__version__} — CUDA available: {torch.cuda.is_available()}')
print('All good.')

Dependencies installed. Verifying torch is still intact...
torch 2.8.0a0+5228986c39.nv25.06 — CUDA available: True
All good.


## Step 2 — Configuration

In [ ]:
import os

DATASET_GDRIVE_ID = '16Jh5qajBB1SZTYNo23WuN9XMbsMqg9CC'   # plate_dataset.zip
MODEL_GDRIVE_ID   = '1rz3F0DYsRFNblsAXvLABaWaxdZP8piB6'   # trocr_model.zip

# ── Local paths (persistent storage on E2E at /home/jovyan) ──────────────────
BASE_DIR         = '/home/jovyan'
DATASET_ZIP      = os.path.join(BASE_DIR, 'plate_dataset.zip')
MODEL_ZIP        = os.path.join(BASE_DIR, 'trocr_model.zip')
DATASET_DIR      = os.path.join(BASE_DIR, 'plate_dataset')       # extracted dataset root
IMAGES_DIR       = os.path.join(DATASET_DIR, 'images')
LABELS_CSV       = os.path.join(DATASET_DIR, 'labels.csv')
PRETRAINED_DIR   = os.path.join(BASE_DIR, 'trocr_model')         # extracted model root
CHECKPOINT_DIR   = os.path.join(BASE_DIR, 'trocr_v2_checkpoints')
FINAL_MODEL_DIR  = os.path.join(BASE_DIR, 'trocr_alpr_v2_final')

os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(FINAL_MODEL_DIR, exist_ok=True)

# ── HuggingFace cache → persistent storage ────────────────────────────────────
HF_CACHE_DIR = os.path.join(BASE_DIR, '.cache', 'huggingface')
os.makedirs(HF_CACHE_DIR, exist_ok=True)
os.environ['HF_HOME']            = HF_CACHE_DIR
os.environ['TRANSFORMERS_CACHE'] = HF_CACHE_DIR
os.environ['HF_DATASETS_CACHE']  = HF_CACHE_DIR

# ── Hyperparameters ───────────────────────────────────────────────────────────
# Lower LR than v1 (5e-5) because we are re-fine-tuning an already trained model.
# Too high LR = catastrophic forgetting of what the model already learned.
TRAIN_BATCH_SIZE = 48      # Safe for A100 80GB with trocr-base
EVAL_BATCH_SIZE  = 96
NUM_EPOCHS       = 10
LEARNING_RATE    = 2e-5    # Lower than v1's 5e-5 — intentional
WARMUP_STEPS     = 300
MAX_TARGET_LEN   = 16      # Indian plates max ~12 chars, 16 is safe buffer
VAL_SPLIT        = 0.1

print('Config ready.')
print(f'  Pretrained model : {PRETRAINED_DIR}')
print(f'  Dataset          : {DATASET_DIR}')
print(f'  Batch size       : {TRAIN_BATCH_SIZE} train / {EVAL_BATCH_SIZE} eval')
print(f'  Epochs           : {NUM_EPOCHS}')
print(f'  Learning rate    : {LEARNING_RATE}  (lower than v1 — intentional)')
print(f'  Max label len    : {MAX_TARGET_LEN}')

Config ready.
  Pretrained model : /home/jovyan/trocr_model
  Dataset          : /home/jovyan/plate_dataset
  Batch size       : 48 train / 96 eval
  Epochs           : 10
  Learning rate    : 2e-05  (lower than v1 — intentional)
  Max label len    : 16


## Step 3 — Download & Extract Dataset


In [4]:
import gdown, zipfile, shutil, glob
from tqdm import tqdm

# ── Download ──────────────────────────────────────────────────────────────────
if os.path.exists(DATASET_ZIP):
    print(f'Zip already exists ({os.path.getsize(DATASET_ZIP)/1024/1024:.1f} MB) — skipping download.')
else:
    print('Downloading plate_dataset.zip from Google Drive...')
    gdown.download(id=DATASET_GDRIVE_ID, output=DATASET_ZIP, quiet=False)

zip_mb = os.path.getsize(DATASET_ZIP) / 1024 / 1024
print(f'Zip size: {zip_mb:.1f} MB')

with zipfile.ZipFile(DATASET_ZIP) as z:
    all_entries = z.namelist()
print(f'Entries in zip: {len(all_entries)}')
print('First 5 entries:')
for e in all_entries[:5]:
    print(f'  {e}')

Downloading...
From (original): https://drive.google.com/uc?id=16Jh5qajBB1SZTYNo23WuN9XMbsMqg9CC
From (redirected): https://drive.google.com/uc?id=16Jh5qajBB1SZTYNo23WuN9XMbsMqg9CC&confirm=t&uuid=67603c94-f280-47bf-9be2-581173a51f49
To: /home/jovyan/plate_dataset.zip
100%|██████████| 544M/544M [00:12<00:00, 43.8MB/s] 


Zip size: 519.1 MB
Entries in zip: 50003
First 5 entries:
  plate_dataset/
  plate_dataset/images/
  plate_dataset/images/plate_00000.jpg
  plate_dataset/images/plate_00001.jpg
  plate_dataset/images/plate_00002.jpg


In [ ]:
# Extract
n_existing = 0
if os.path.exists(IMAGES_DIR):
    n_existing = len([f for f in os.listdir(IMAGES_DIR)
                      if f.lower().endswith(('.jpg', '.jpeg', '.png'))])

if n_existing > 0 and os.path.exists(LABELS_CSV):
    print(f'Already extracted — {n_existing} images found. Skipping.')
else:
    STAGE = os.path.join(BASE_DIR, '_dataset_stage')
    if os.path.exists(STAGE):
        shutil.rmtree(STAGE)
    os.makedirs(STAGE)

    print('Extracting...')
    with zipfile.ZipFile(DATASET_ZIP) as z:
        members = z.namelist()
        for m in tqdm(members, desc='Extracting'):
            z.extract(m, STAGE)

    images_found = glob.glob(os.path.join(STAGE, '**/images'), recursive=True)
    csv_found    = glob.glob(os.path.join(STAGE, '**/labels.csv'), recursive=True)
    print(f'images/ folder found : {images_found}')
    print(f'labels.csv found     : {csv_found}')

    os.makedirs(DATASET_DIR, exist_ok=True)

    if images_found and csv_found:
        if not os.path.exists(IMAGES_DIR):
            shutil.copytree(images_found[0], IMAGES_DIR)
        shutil.copy2(csv_found[0], LABELS_CSV)
    else:
        # Flat zip — no images/ subfolder
        csv_files = glob.glob(os.path.join(STAGE, '**/*.csv'), recursive=True)
        if not csv_files:
            raise FileNotFoundError('labels.csv not found in zip!')
        shutil.copy2(csv_files[0], LABELS_CSV)
        os.makedirs(IMAGES_DIR, exist_ok=True)
        moved = 0
        for ext in ('*.jpg', '*.jpeg', '*.png'):
            for f in glob.glob(os.path.join(STAGE, '**', ext), recursive=True):
                shutil.copy2(f, os.path.join(IMAGES_DIR, os.path.basename(f)))
                moved += 1
        print(f'Flat layout: moved {moved} images.')

    shutil.rmtree(STAGE)
    print('Staging area cleaned up.')

# Verify
n_images = len([f for f in os.listdir(IMAGES_DIR)
                if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
print(f'\nImages found : {n_images}')
print(f'Labels CSV   : {LABELS_CSV} — exists: {os.path.exists(LABELS_CSV)}')

if n_images == 0:
    raise RuntimeError('No images found after extraction!')
if not os.path.exists(LABELS_CSV):
    raise FileNotFoundError('labels.csv not found after extraction!')

print('\nDataset ready.')

Extracting...


Extracting: 100%|██████████| 50003/50003 [00:05<00:00, 9206.11it/s]


images/ folder found : ['/home/jovyan/_dataset_stage/plate_dataset/images']
labels.csv found     : ['/home/jovyan/_dataset_stage/plate_dataset/labels.csv']
Staging area cleaned up.

Images found : 50000
Labels CSV   : /home/jovyan/plate_dataset/labels.csv — exists: True

Dataset ready.


## Step 4 — Download & Extract Your Fine-tuned Model
Downloads `trocr_model.zip` and extracts your model weights to `/home/jovyan/trocr_model/`

In [ ]:
# Download model zip
if os.path.exists(MODEL_ZIP):
    print(f'Model zip already exists ({os.path.getsize(MODEL_ZIP)/1024/1024:.1f} MB) — skipping download.')
else:
    print('Downloading trocr_model.zip from Google Drive...')
    gdown.download(id=MODEL_GDRIVE_ID, output=MODEL_ZIP, quiet=False)

model_zip_mb = os.path.getsize(MODEL_ZIP) / 1024 / 1024
print(f'Model zip size: {model_zip_mb:.1f} MB')

with zipfile.ZipFile(MODEL_ZIP) as z:
    model_entries = z.namelist()
print(f'Entries in model zip: {len(model_entries)}')
for e in model_entries:
    print(f'  {e}')

Downloading...
From (original): https://drive.google.com/uc?id=1rz3F0DYsRFNblsAXvLABaWaxdZP8piB6
From (redirected): https://drive.google.com/uc?id=1rz3F0DYsRFNblsAXvLABaWaxdZP8piB6&confirm=t&uuid=13012b72-fdfb-450f-a872-07a1034e48bf
To: /home/jovyan/trocr_model.zip
100%|██████████| 1.24G/1.24G [00:29<00:00, 42.4MB/s]

Model zip size: 1180.9 MB
Entries in model zip: 12
  trocr_model/
  trocr_model/config.json
  trocr_model/generation_config.json
  trocr_model/merges.txt
  trocr_model/model.safetensors
  trocr_model/preprocessor_config.json
  trocr_model/special_tokens_map.json
  trocr_model/tokenizer.json
  trocr_model/tokenizer_config.json
  trocr_model/training_args.bin
  trocr_model/trocr.txt
  trocr_model/vocab.json


In [ ]:
# Extract model zip

REQUIRED_FILES = ['model.safetensors', 'config.json', 'preprocessor_config.json']

already_ok = (
    os.path.exists(PRETRAINED_DIR) and
    all(os.path.exists(os.path.join(PRETRAINED_DIR, f)) for f in REQUIRED_FILES)
)

if already_ok:
    print(f'Model already extracted at {PRETRAINED_DIR} — skipping.')
else:
    MODEL_STAGE = os.path.join(BASE_DIR, '_model_stage')
    if os.path.exists(MODEL_STAGE):
        shutil.rmtree(MODEL_STAGE)
    os.makedirs(MODEL_STAGE)

    print('Extracting model zip...')
    with zipfile.ZipFile(MODEL_ZIP) as z:
        for m in tqdm(z.namelist(), desc='Extracting model'):
            z.extract(m, MODEL_STAGE)

    # Find where model.safetensors landed
    safetensors_hits = glob.glob(
        os.path.join(MODEL_STAGE, '**/model.safetensors'), recursive=True
    )
    if not safetensors_hits:
        raise FileNotFoundError(
            'model.safetensors not found in zip! '
            f'Zip contents: {os.listdir(MODEL_STAGE)}'
        )

    # The folder containing model.safetensors is our model dir
    extracted_model_dir = os.path.dirname(safetensors_hits[0])
    print(f'Found model dir inside zip: {extracted_model_dir}')

    if os.path.exists(PRETRAINED_DIR):
        shutil.rmtree(PRETRAINED_DIR)
    shutil.copytree(extracted_model_dir, PRETRAINED_DIR)
    shutil.rmtree(MODEL_STAGE)
    print('Model staging cleaned up.')

# Verify
print(f'\nModel files at {PRETRAINED_DIR}:')
for f in sorted(os.listdir(PRETRAINED_DIR)):
    size_mb = os.path.getsize(os.path.join(PRETRAINED_DIR, f)) / 1024 / 1024
    print(f'  {f:45s} {size_mb:.1f} MB')

for req in REQUIRED_FILES:
    path = os.path.join(PRETRAINED_DIR, req)
    if not os.path.exists(path):
        raise FileNotFoundError(f'Required file missing: {req}')

print('\nModel extraction verified.')

Extracting model zip...


Extracting model: 100%|██████████| 12/12 [00:05<00:00,  2.20it/s]


Found model dir inside zip: /home/jovyan/_model_stage/trocr_model
Model staging cleaned up.

Model files at /home/jovyan/trocr_model:
  config.json                                   0.0 MB
  generation_config.json                        0.0 MB
  merges.txt                                    0.4 MB
  model.safetensors                             1273.9 MB
  preprocessor_config.json                      0.0 MB
  special_tokens_map.json                       0.0 MB
  tokenizer.json                                2.0 MB
  tokenizer_config.json                         0.0 MB
  training_args.bin                             0.0 MB
  trocr.txt                                     0.0 MB
  vocab.json                                    0.8 MB

Model extraction verified.


## Step 5 — Load & Validate Labels

In [ ]:
import pandas as pd

df = pd.read_csv(LABELS_CSV, dtype=str)
df.columns = [c.strip().lower() for c in df.columns]
print(f'CSV columns: {list(df.columns)}')

# Auto-detect filename and label columns

filename_col = next(
    (c for c in df.columns if 'file' in c or 'image' in c or 'name' in c),
    df.columns[0]
)
plate_col = next(
    (c for c in df.columns if 'label' in c or 'plate' in c or 'text' in c),
    df.columns[1]
)
df = df.rename(columns={filename_col: 'filename', plate_col: 'plate'})
print(f'Mapped columns -> filename: "{filename_col}", label: "{plate_col}"')

# Clean
df['filename'] = df['filename'].str.strip()
df['plate']    = df['plate'].str.strip().str.upper()
df.dropna(subset=['filename', 'plate'], inplace=True)

# Drop rows where the image file is missing on disk
df['exists'] = df['filename'].apply(
    lambda f: os.path.exists(os.path.join(IMAGES_DIR, f))
)
n_missing = (~df['exists']).sum()
if n_missing > 0:
    print(f'WARNING: {n_missing} rows have no matching image file. Dropping.')
df = df[df['exists']].drop(columns=['exists']).reset_index(drop=True)

# Drop plates that are clearly too long (data error)
df = df[df['plate'].str.len() <= MAX_TARGET_LEN].reset_index(drop=True)

print(f'\nValid samples : {len(df)}')
print('Plate length distribution:')
print(df['plate'].str.len().value_counts().sort_index().to_string())
print('\nSample rows:')
print(df.sample(5, random_state=42)[['filename', 'plate']].to_string(index=False))

CSV columns: ['filename', 'label', 'raw_text']
Mapped columns -> filename: "filename", label: "label"

Valid samples : 50000
Plate length distribution:
plate
8      3656
9      9059
10    26176
11     7137
12     3972

Sample rows:
       filename        plate
plate_33553.jpg   LD17E03364
plate_09427.jpg   UT11EF0547
plate_00199.jpg JK49TEMP8240
plate_12447.jpg    21BH0287Q
plate_39489.jpg  40CCAG44839


## Step 6 — Train / Val Split

In [9]:
from sklearn.model_selection import train_test_split

df_train, df_val = train_test_split(
    df, test_size=VAL_SPLIT, random_state=42, shuffle=True
)
df_train = df_train.reset_index(drop=True)
df_val   = df_val.reset_index(drop=True)

print(f'Train : {len(df_train)} samples')
print(f'Val   : {len(df_val)} samples')

Train : 45000 samples
Val   : 5000 samples


## Step 7 — Load Your Fine-tuned Model & Processor

In [10]:
import torch
from transformers import TrOCRProcessor, VisionEncoderDecoderModel

print(f'Loading model from: {PRETRAINED_DIR}')
processor = TrOCRProcessor.from_pretrained(PRETRAINED_DIR)
model     = VisionEncoderDecoderModel.from_pretrained(PRETRAINED_DIR)

# Token ID config — must match what was set during v1 fine-tuning
model.config.decoder_start_token_id = processor.tokenizer.cls_token_id
model.config.pad_token_id           = processor.tokenizer.pad_token_id
model.config.eos_token_id           = processor.tokenizer.sep_token_id
model.config.vocab_size             = model.config.decoder.vocab_size
model.config.max_new_tokens         = MAX_TARGET_LEN

device = torch.device('cuda')
model.to(device)

params_m = sum(p.numel() for p in model.parameters()) / 1e6
print(f'Model loaded   : {params_m:.1f}M parameters')
print(f'Device         : {device} ({torch.cuda.get_device_name(0)})')
print(f'VRAM used      : {torch.cuda.memory_allocated(0)/1e9:.2f} GB')
print(f'BF16 supported : {torch.cuda.is_bf16_supported()}')
print(f'decoder_start  : {model.config.decoder_start_token_id}')
print(f'pad_token_id   : {model.config.pad_token_id}')
print(f'eos_token_id   : {model.config.eos_token_id}')

/usr/local/lib/python3.12/dist-packages/transformers/utils/hub.py:127: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


Loading model from: /home/jovyan/trocr_model
Model loaded   : 333.9M parameters
Device         : cuda (NVIDIA A100 80GB PCIe)
VRAM used      : 1.35 GB
BF16 supported : True
decoder_start  : 0
pad_token_id   : 1
eos_token_id   : 2


## Step 8 — Dataset Class & CER Metric

In [ ]:
import warnings
import evaluate
import numpy as np
from torch.utils.data import Dataset
from PIL import Image

# Dataset class
class PlateDataset(Dataset):
    def __init__(self, df, images_dir, processor, max_target_len):
        self.df            = df
        self.images_dir    = images_dir
        self.processor     = processor
        self.max_target_len = max_target_len

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = os.path.join(self.images_dir, row['filename'])
        image = Image.open(img_path).convert('RGB')

        pixel_values = self.processor(
            image, return_tensors='pt'
        ).pixel_values.squeeze(0)

        with warnings.catch_warnings():
            warnings.simplefilter('ignore')
            labels = self.processor.tokenizer(
                text_target=row['plate'],
                padding='max_length',
                max_length=self.max_target_len,
                truncation=True,
                return_tensors='pt',
            ).input_ids.squeeze(0)

        # Replace padding token id with -100 so loss ignores padding
        labels[labels == self.processor.tokenizer.pad_token_id] = -100

        return {'pixel_values': pixel_values, 'labels': labels}


# CER metric
cer_metric = evaluate.load('cer')

def compute_metrics(pred):
    label_ids  = pred.label_ids
    pred_ids   = pred.predictions

    # Replace -100 (padding) back to pad_token_id for decoding
    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id

    pred_str  = processor.batch_decode(pred_ids,   skip_special_tokens=True)
    label_str = processor.batch_decode(label_ids,  skip_special_tokens=True)

    # Exact match accuracy
    exact = np.mean([
        p.strip().upper() == l.strip().upper()
        for p, l in zip(pred_str, label_str)
    ])

    cer = cer_metric.compute(predictions=pred_str, references=label_str)
    return {'cer': round(cer, 4), 'exact_match': round(float(exact), 4)}


# Build datasets 
train_dataset = PlateDataset(df_train, IMAGES_DIR, processor, MAX_TARGET_LEN)
val_dataset   = PlateDataset(df_val,   IMAGES_DIR, processor, MAX_TARGET_LEN)

print(f'Train dataset : {len(train_dataset)} samples')
print(f'Val dataset   : {len(val_dataset)} samples')

# Quick test
sample = train_dataset[0]
print(f'\nSmoke test passed:')
print(f'  pixel_values shape : {sample["pixel_values"].shape}')
print(f'  labels shape       : {sample["labels"].shape}')

Train dataset : 45000 samples
Val dataset   : 5000 samples

Smoke test passed:
  pixel_values shape : torch.Size([3, 384, 384])
  labels shape       : torch.Size([16])


## Step 9 — Cost Estimate

In [ ]:
steps_per_epoch = len(train_dataset) // TRAIN_BATCH_SIZE
total_steps     = steps_per_epoch * NUM_EPOCHS

# Rough estimate: ~0.25s per step on A100 80GB for trocr-base at batch 48
est_seconds = total_steps * 0.25
est_hours   = est_seconds / 3600
est_cost_rs = est_hours * 180

print('=' * 50)
print('COST ESTIMATE (A100 80GB @ ₹180/hr)')
print('=' * 50)
print(f'  Training samples  : {len(train_dataset)}')
print(f'  Batch size        : {TRAIN_BATCH_SIZE}')
print(f'  Steps per epoch   : {steps_per_epoch}')
print(f'  Total steps       : {total_steps}')
print(f'  Epochs            : {NUM_EPOCHS}')
print(f'  Estimated time    : {est_hours:.1f} hours')
print(f'  Estimated cost    : ₹{est_cost_rs:.0f}')
print('=' * 50)
print('Actual time may vary by 20%. Add ~15 min for setup overhead.')
print('Make sure you have enough credits before proceeding.')

COST ESTIMATE (A100 80GB @ ₹180/hr)
  Training samples  : 45000
  Batch size        : 48
  Steps per epoch   : 937
  Total steps       : 9370
  Epochs            : 10
  Estimated time    : 0.7 hours
  Estimated cost    : ₹117
NOTE: Actual time may vary ±20%. Add ~15 min for setup overhead.
Make sure you have enough credits before proceeding.


## Step 10 — Training Arguments

In [13]:
from transformers import Seq2SeqTrainingArguments

training_args = Seq2SeqTrainingArguments(
    output_dir=CHECKPOINT_DIR,

    # Core
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    warmup_steps=WARMUP_STEPS,
    weight_decay=0.01,

    # Precision — BF16 is faster and more stable than FP16 on A100
    bf16=torch.cuda.is_bf16_supported(),
    fp16=False,

    # Evaluation and saving
    evaluation_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='cer',
    greater_is_better=False,       # Lower CER = better
    save_total_limit=2,            # Keep only 2 checkpoints to save disk

    # Generation — required for Seq2Seq eval
    predict_with_generate=True,
    generation_max_length=MAX_TARGET_LEN,

    # Logging
    logging_dir=os.path.join(CHECKPOINT_DIR, 'logs'),
    logging_steps=100,
    report_to='none',              # No wandb/tensorboard needed

    # DataLoader
    dataloader_num_workers=4,
    dataloader_pin_memory=True,
)

print('Training arguments set.')
print(f'  BF16 enabled : {training_args.bf16}')
print(f'  Output dir   : {CHECKPOINT_DIR}')

Training arguments set.
  BF16 enabled : True
  Output dir   : /home/jovyan/trocr_v2_checkpoints


/usr/local/lib/python3.12/dist-packages/transformers/training_args.py:1525: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


## Step 11 — Build Trainer

In [14]:
from transformers import Seq2SeqTrainer, default_data_collator

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
    data_collator=default_data_collator,
    tokenizer=processor.feature_extractor,
)

print('Trainer ready.')
print(f'  Train batches per epoch : {len(trainer.get_train_dataloader())}')
print(f'  Val batches per epoch   : {len(trainer.get_eval_dataloader())}')

Trainer ready.
  Train batches per epoch : 938
  Val batches per epoch   : 53


/usr/local/lib/python3.12/dist-packages/transformers/models/trocr/processing_trocr.py:137: FutureWarning: `feature_extractor` is deprecated and will be removed in v5. Use `image_processor` instead.
  warnings.warn(


## Step 12 — Train

In [15]:
import time

print('Starting training...')
print(f'VRAM before training : {torch.cuda.memory_allocated(0)/1e9:.2f} GB')

t0 = time.time()
train_result = trainer.train()
elapsed = time.time() - t0

print(f'\nTraining complete.')
print(f'  Total time    : {elapsed/3600:.2f} hours')
print(f'  Cost estimate : ₹{elapsed/3600 * 180:.0f}')
print(f'  Train loss    : {train_result.training_loss:.4f}')
print(f'  Steps done    : {train_result.global_step}')

Starting training...
VRAM before training : 1.35 GB


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The 

Epoch,Training Loss,Validation Loss,Cer,Exact Match
1,0.161100,0.165204,0.030200,0.827800
2,0.110200,0.123396,0.023200,0.864000
3,0.064300,0.112114,0.021800,0.871800
4,0.042600,0.106655,0.018800,0.887200
5,0.028400,0.101137,0.017100,0.895600
6,0.014800,0.101525,0.016400,0.901600
7,0.006800,0.094626,0.015300,0.909200
8,0.002400,0.095088,0.014600,0.912800
9,0.001600,0.096769,0.014200,0.913200
10,0.001500,0.095400,0.013700,0.917000


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The 


Training complete.
  Total time    : 1.18 hours
  Cost estimate : ₹212
  Train loss    : 0.0777
  Steps done    : 9380


## Step 13 — Evaluate on Validation Set

In [16]:
print('Running final evaluation on validation set...')
metrics = trainer.evaluate()

print('\n=== Final Validation Metrics ===')
for k, v in metrics.items():
    print(f'  {k:30s} : {v}')

cer = metrics.get('eval_cer', 1.0)
em  = metrics.get('eval_exact_match', 0.0)

print('\n=== Interpretation ===')
print(f'  CER         : {cer:.4f}  (lower is better, 0.0 = perfect)')
print(f'  Exact match : {em:.4f}  (higher is better, 1.0 = perfect)')

if cer < 0.02:
    print('  Result: Excellent — production ready')
elif cer < 0.05:
    print('  Result: Very good — minor errors on hard cases only')
elif cer < 0.15:
    print('  Result: Good — consider more epochs or data augmentation')
else:
    print('  Result: Needs work — check data quality and training config')

Running final evaluation on validation set...


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The 


=== Final Validation Metrics ===
  eval_loss                      : 0.09539997577667236
  eval_cer                       : 0.0137
  eval_exact_match               : 0.917
  eval_runtime                   : 44.9214
  eval_samples_per_second        : 111.306
  eval_steps_per_second          : 1.18
  epoch                          : 10.0

=== Interpretation ===
  CER         : 0.0137  (lower is better, 0.0 = perfect)
  Exact match : 0.9170  (higher is better, 1.0 = perfect)
  Result: Excellent — production ready


## Step 14 — Save Model

In [17]:
# Saves the best checkpoint (load_best_model_at_end=True loaded it automatically)
trainer.save_model(FINAL_MODEL_DIR)
processor.save_pretrained(FINAL_MODEL_DIR)

print(f'Model saved to: {FINAL_MODEL_DIR}')
print('\nFiles saved:')
total_mb = 0
for f in sorted(os.listdir(FINAL_MODEL_DIR)):
    size_mb = os.path.getsize(os.path.join(FINAL_MODEL_DIR, f)) / 1024 / 1024
    total_mb += size_mb
    print(f'  {f:45s} {size_mb:.1f} MB')
print(f'\nTotal size : {total_mb:.1f} MB')
print(f'Location   : {FINAL_MODEL_DIR} (persistent storage — survives restarts)')

Model saved to: /home/jovyan/trocr_alpr_v2_final

Files saved:
  config.json                                   0.0 MB
  generation_config.json                        0.0 MB
  merges.txt                                    0.4 MB
  model.safetensors                             1273.9 MB
  preprocessor_config.json                      0.0 MB
  special_tokens_map.json                       0.0 MB
  tokenizer.json                                2.0 MB
  tokenizer_config.json                         0.0 MB
  training_args.bin                             0.0 MB
  vocab.json                                    0.8 MB

Total size : 1277.1 MB
Location   : /home/jovyan/trocr_alpr_v2_final (persistent storage — survives restarts)


## Step 15 — Zip & Backup Model to JupyterLab

In [18]:
import zipfile

BACKUP_ZIP = os.path.join(BASE_DIR, 'trocr_alpr_v2_final.zip')
print('Zipping model for backup...')

with zipfile.ZipFile(BACKUP_ZIP, 'w', zipfile.ZIP_DEFLATED) as zf:
    for f in sorted(os.listdir(FINAL_MODEL_DIR)):
        zf.write(os.path.join(FINAL_MODEL_DIR, f), arcname=f)

zip_mb = os.path.getsize(BACKUP_ZIP) / 1024 / 1024
print(f'Zip size : {zip_mb:.1f} MB')
print(f'Zip path : {BACKUP_ZIP}')
print('Download it from the JupyterLab file browser (left panel).')

Zipping model for backup...
Zip size : 1181.1 MB
Zip path : /home/jovyan/trocr_alpr_v2_final.zip
Download it from the JupyterLab file browser (left panel).


## Step 16 — Inference Test
Quick sanity check — runs prediction on 10 random validation samples.

In [19]:
from transformers import TrOCRProcessor, VisionEncoderDecoderModel
from PIL import Image

inf_processor = TrOCRProcessor.from_pretrained(FINAL_MODEL_DIR)
inf_model     = VisionEncoderDecoderModel.from_pretrained(FINAL_MODEL_DIR).to(device)
inf_model.eval()

sample_rows = df_val.sample(10, random_state=42).reset_index(drop=True)

print('=== Inference on 10 val samples ===')
correct = 0
for _, row in sample_rows.iterrows():
    img_path = os.path.join(IMAGES_DIR, row['filename'])
    img      = Image.open(img_path).convert('RGB')
    pv       = inf_processor(img, return_tensors='pt').pixel_values.to(device)
    with torch.no_grad():
        ids = inf_model.generate(pv, max_new_tokens=MAX_TARGET_LEN)
    pred  = inf_processor.batch_decode(ids, skip_special_tokens=True)[0].strip().upper()
    label = row['plate'].strip().upper()
    match = pred == label
    correct += int(match)
    status = 'OK' if match else 'X '
    print(f'  [{status}] Pred: {pred:15s} | Label: {label}')

print(f'\nExact match: {correct}/10')

=== Inference on 10 val samples ===
  [OK] Pred: CG50IO1093      | Label: CG50IO1093
  [OK] Pred: LA802QN2155     | Label: LA802QN2155
  [OK] Pred: 22BH9028T       | Label: 22BH9028T
  [OK] Pred: CH96EG4721      | Label: CH96EG4721
  [OK] Pred: AR38SML6441     | Label: AR38SML6441
  [OK] Pred: HP79ZV3379      | Label: HP79ZV3379
  [OK] Pred: NL06U85506      | Label: NL06U85506
  [OK] Pred: MZ68DR6228      | Label: MZ68DR6228
  [OK] Pred: 55CC0095        | Label: 55CC0095
  [X ] Pred: RJ02OO6427      | Label: RJ02O06427

Exact match: 9/10
